# Banking77 v0.1: one free GPU, single-seed pilot
Use Runtime → Change runtime type → T4 GPU. Free allocation is not guaranteed.
Add HF_TOKEN in Colab Secrets and enable notebook access. Never paste it into a cell.
This runner uses 200 balanced teacher sources, all 8,992 gold rows, seed 1 for both students,
and all 3,080 official test queries. It measures TF-IDF, zero-shot 7B and two students.
Remaining seeds, few-shot comparison and full-source distillation remain future work.
Private checkpoint uploads are downloaded and checksum-verified before training and after stages.
Publishing is authorized but blocked until the measured evidence passes. Runtime is not guaranteed;
no paid compute is provisioned and no keep-alive bypass is used.


In [ ]:
import os
import pathlib
import subprocess
import sys

root = pathlib.Path('/content/Banking77Distillation')
if not root.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jatinmallarapu/Banking77Distillation.git', str(root)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.13.9'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.11', '/content/b77-venv'], check=True)
gpu_python = '/content/b77-venv/bin/python'
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', gpu_python,
                '-e', str(root) + '[train,teacher,teacher-eval,dev]'], check=True)
gpu_env = dict(os.environ, PATH='/content/b77-venv/bin:' + os.environ['PATH'], HF_HOME='/content/huggingface')
gpu_env.pop('HF_TOKEN', None)
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
from google.colab import userdata

# A foreground process owns the stages. The secret never appears in arguments or output.
subprocess.run([gpu_python, '-m', 'b77.colab_pipeline', '--token-stdin',
                '--declare-single-seed-pilot', '--max-hours', '10'],
               cwd=root, env=gpu_env, text=True, check=True,
               input=userdata.get('HF_TOKEN'))


## Recovery after a disconnect
Do not start over and overwrite the private backup. In a fresh runtime, install the dependencies
using the setup cell, then run the recovery cell below instead of the first-launch cell.
Restoration downloads the immutable backup commit and verifies its checksum before extraction.
Complete training checkpoints resume with optimizer state; completed adapters are reused.
Incomplete evaluation stages are rerun. Stage logs are in runs/. Never display the secret.


In [ ]:
# Recovery only: run after a disconnect, not during the first launch.
# from google.colab import userdata
# recovery = pathlib.Path('/content/Banking77Recovered')
# subprocess.run([gpu_python, '-m', 'b77.backup', 'restore', '--token-stdin',
#                 '--destination', str(recovery)], cwd=root, env=gpu_env, text=True,
#                check=True, input=userdata.get('HF_TOKEN'))
# subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', gpu_python,
#                 '--reinstall-package', 'banking77-distill', '-e', str(recovery)], check=True)
# subprocess.run([gpu_python, '-m', 'b77.colab_pipeline', '--token-stdin', '--max-hours', '10'],
#                cwd=recovery, env=gpu_env, text=True, check=True, input=userdata.get('HF_TOKEN'))
